# STUDYTRACK IA
## TED 03 — Modelagem Preditiva e Machine Learning
### Classificação multiclasse do desempenho acadêmico

**Instituição:** Centro Universitário de Balsas — UNIBALSAS  
**Curso:** Análise e Desenvolvimento de Sistemas  
**Disciplina:** Big Data e Ciências de Dados — 2026.2  
**Professor:** Alex Rogaleski Marques  
**Equipe:** [confirmar identificação e participação dos integrantes]  
**Versão:** caderno técnico da TED 03, em validação pelo grupo

> **Situação:** metodologia confirmada; a comparação dos modelos será calculada pelo código. A escolha final, a avaliação do teste e as conclusões dependem da revisão do grupo.

**Questão de pesquisa:** considerando as 31 características do conjunto *Higher Education Students Performance Evaluation*, como os classificadores Regressão Logística, Árvore de Decisão e Random Forest se comparam na previsão da classe final `grade`?

**Escopo do notebook:** preparação para modelagem, divisão de dados, pipelines independentes, validação cruzada, comparação, decisão do grupo, teste final e interpretação orientada. Os resultados não devem ser apresentados como conclusões aprovadas antes da conferência.


## 1. Introdução e continuidade das TEDs

O StudyTrack IA pretende auxiliar estudantes universitários na organização de disciplinas, atividades, avaliações e prazos. Na **TED 01**, o grupo selecionou uma base pública sobre hábitos e desempenho acadêmico; na **TED 02**, verificou a qualidade dessa base, padronizou os nomes das variáveis e desenvolveu uma análise exploratória.

Entre os achados registrados na TED 02 estão: 145 registros, ausência de dados faltantes e duplicados, presença de oito classes de desempenho e maior associação descritiva do histórico acadêmico com a nota final do que do tempo de estudo isolado. Esses resultados motivam a presente etapa, mas **associação não demonstra causalidade**.

A **TED 03** investiga a capacidade de classificação dos dados disponíveis. Ela não cria automaticamente o mecanismo de priorização de tarefas do MVP, pois a base não contém prazos, pesos de avaliações, situação das atividades ou calendário acadêmico.

**Documentos anteriores:** [TED 01](https://github.com/Francisco-alt-coder/StudyTrack_IA/blob/main/TED_01_Big_Data_StudyTrack_IA.pdf) · [TED 02](https://github.com/Francisco-alt-coder/StudyTrack_IA/blob/main/TED02_StudyTrack_.pdf).


## 2. Problema de modelagem e decisões do grupo

A variável **`grade`** representa uma de oito classes: 0 = Fail, 1 = DD, 2 = DC, 3 = CC, 4 = CB, 5 = BB, 6 = BA, 7 = AA. O objetivo é **classificação multiclasse**, e não previsão de um valor contínuo.

| Decisão metodológica | Configuração aprovada |
|---|---|
| Variável-alvo | `grade` |
| Preditoras | 31, sem `student_id` |
| Divisão reservada | 80% treinamento e 20% teste, com estratificação |
| Semente | 42 |
| Validação cruzada | `StratifiedKFold`, cinco partes, apenas no treino |
| Algoritmos | Regressão Logística, Árvore de Decisão, Random Forest |
| Pré-processamento | 16 variáveis nominais por One-Hot; 15 ordinais com códigos ordenados |
| Padronização | Apenas das ordinais na Regressão Logística |
| Seleção | F1-macro médio da validação cruzada; acurácia como apoio |
| Avaliação final | Matriz de confusão, F1-macro, acurácia e relatório por classe |

**Justificativa das métricas:** o F1-macro calcula o F1 de cada classe e depois uma média com pesos iguais. Isso evita que o indicador principal seja dominado pelas classes mais frequentes. A acurácia mede a proporção global de acertos. Com poucas observações por classe, ambos os resultados devem ser interpretados com cautela.

**Regra de integridade:** o conjunto de teste fica reservado até o grupo registrar a escolha do classificador. Não se deve escolher o modelo com base no seu resultado no teste.


## 3. Fonte de dados, bibliotecas e inspeção

**Fonte original:** YILMAZ, N.; ŞEKEROĞLU, B. *Higher Education Students Performance Evaluation*. UCI Machine Learning Repository. DOI: [10.24432/C51G82](https://doi.org/10.24432/C51G82).

**Arquivo utilizado:** `data/processed/dataset_modelagem.csv`, derivado da TED 02. O endereço abaixo aponta para uma revisão específica do repositório para garantir reprodutibilidade. O arquivo original permanece preservado em `data/raw/`.

**Dependências:** Python, Pandas, NumPy, Matplotlib e Scikit-learn. O Google Colab normalmente fornece essas bibliotecas; registrar as versões efetivamente utilizadas.


In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    ConfusionMatrixDisplay
)

SEMENTE = 42
plt.rcParams.update({"figure.figsize": (8, 4.8), "axes.grid": False, "font.size": 10})

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)


In [ ]:
URL_DADOS = (
    "https://raw.githubusercontent.com/Francisco-alt-coder/"
    "StudyTrack_IA/61d910e73b19f868b9e234cbd92c8756216a53a9/"
    "data/processed/dataset_modelagem.csv"
)

dados = pd.read_csv(URL_DADOS)
print("Dimensão da base:", dados.shape)
display(dados.head(3))


In [ ]:
alvo = "grade"
assert alvo in dados.columns, "A coluna-alvo não foi encontrada."

X = dados.drop(columns=[alvo])
y = dados[alvo]

assert X.shape[1] == 31, "As 31 variáveis aprovadas não foram encontradas."
assert set(y.unique()) == set(range(8)), "As classes de grade diferem das documentadas."

qualidade = pd.DataFrame({
    "verificacao": ["Registros", "Preditoras", "Classes", "Valores ausentes", "Duplicados"],
    "resultado": [
        len(dados), X.shape[1], y.nunique(),
        int(dados.isna().sum().sum()), int(dados.duplicated().sum())
    ]
})
display(qualidade)
print("Tipos de dados presentes:", X.dtypes.value_counts().to_dict())


### 3.1 Distribuição da variável-alvo

A desigualdade entre as classes foi apontada na TED 02 e pode aumentar a instabilidade da avaliação. A figura a seguir é calculada novamente a partir da base de modelagem, mantendo rastreabilidade com a etapa anterior.


In [ ]:
rotulos_grade = {0: "Fail", 1: "DD", 2: "DC", 3: "CC", 4: "CB", 5: "BB", 6: "BA", 7: "AA"}
frequencias = y.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 4.6))
barras = ax.bar(
    [rotulos_grade[classe] for classe in frequencias.index],
    frequencias.values, color="#3869a6"
)
ax.bar_label(barras, padding=3)
ax.set_title("Distribuição das oito classes de desempenho")
ax.set_xlabel("Classe de grade")
ax.set_ylabel("Número de estudantes")
ax.set_ylim(0, int(frequencias.max()) + 7)
fig.tight_layout()
plt.show()

display(pd.DataFrame({
    "classe": [rotulos_grade[c] for c in frequencias.index],
    "registros": frequencias.values,
    "percentual": (frequencias.values / len(y) * 100).round(1)
}))


## 4. Divisão estratificada entre treinamento e teste

A separação será executada **antes** de qualquer ajuste de transformação. A opção `stratify=y` busca preservar as proporções das oito classes e a semente 42 permite reproduzir a divisão.

Apenas o conjunto de **treinamento** será usado para comparar classificadores por validação cruzada. O teste é uma avaliação final, não uma ferramenta de seleção. Como a menor classe possui poucos registros, haverá apenas um pequeno número de exemplos dessa classe em cada parte.


In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEMENTE,
    stratify=y
)

assert set(X_treino.index).isdisjoint(set(X_teste.index))
assert len(X_treino) + len(X_teste) == len(X)
print(f"Treinamento: {len(X_treino)} registros")
print(f"Teste reservado: {len(X_teste)} registros")


In [ ]:
distribuicao = pd.DataFrame({
    "base_total": y.value_counts().sort_index(),
    "treinamento": y_treino.value_counts().sort_index(),
    "teste_reservado": y_teste.value_counts().sort_index()
}).fillna(0).astype(int)

distribuicao.index.name = "classe_grade"
display(distribuicao)
print("Menor classe no treinamento:", int(y_treino.value_counts().min()))


## 5. Representação das variáveis e prevenção de vazamento

Na documentação da UCI, os números são códigos de categorias. O grupo aprovou:

- **16 nominais:** sem ordem natural. Aplicar `OneHotEncoder(handle_unknown="ignore")`, ajustado apenas com cada parte de treinamento.
- **15 ordinais:** mantêm a ordenação documentada. Na Regressão Logística, usar `StandardScaler`; nas árvores, preservar os códigos.

**Limitação metodológica:** manter números para ordinais introduz uma aproximação de distância entre níveis, especialmente para modelos lineares. Essa hipótese deve constar no Documento Técnico.

**Vazamento:** conversões e padronizações fazem parte dos `Pipeline`/`ColumnTransformer`; nunca são ajustadas previamente usando a base completa. O identificador de estudante já foi excluído na TED 02.


In [ ]:
variaveis_nominais = [
    "sexo", "tipo_ensino_medio", "trabalho_adicional",
    "atividade_artistica_esportiva", "possui_parceiro",
    "transporte_universidade", "tipo_moradia", "situacao_pais",
    "ocupacao_mae", "ocupacao_pai", "seminarios_conferencias",
    "impacto_projetos_atividades", "preparacao_prova_companhia",
    "preparacao_prova_momento", "sala_invertida", "course_id"
]

variaveis_ordinais = [
    "idade", "tipo_bolsa", "faixa_renda", "escolaridade_mae",
    "escolaridade_pai", "numero_irmaos", "horas_estudo_semanais",
    "leitura_nao_cientifica", "leitura_cientifica",
    "frequencia_aulas", "anotacoes_aula", "atencao_aula",
    "discussao_interesse_sucesso", "gpa_semestre_anterior",
    "gpa_esperado_formatura"
]

assert len(variaveis_nominais) == 16
assert len(variaveis_ordinais) == 15
assert not (set(variaveis_nominais) & set(variaveis_ordinais))
assert set(variaveis_nominais + variaveis_ordinais) == set(X.columns)

print("Mapeamento aprovado: 16 nominais e 15 ordinais.")


## 6. Construção dos três classificadores

Os três classificadores serão comparados com a mesma divisão, os mesmos atributos e as mesmas métricas. Cada um terá um **pipeline independente**, inclusive quando compartilhar o tipo de codificação das variáveis nominais.

| Modelo | Papel na comparação | Pré-processamento |
|---|---|---|
| Regressão Logística | Referência linear regularizada | One-Hot + padronização ordinal |
| Árvore de Decisão | Regras de decisão interpretáveis | One-Hot + ordinais sem escala |
| Random Forest | Conjunto de árvores com agregação | One-Hot + ordinais sem escala |

Nesta versão, os modelos utilizam parâmetros iniciais de referência, sem busca automática de hiperparâmetros. A Regressão Logística terá limite ampliado de iterações para reduzir falhas de convergência. A escolha e eventuais ajustes posteriores precisam ser discutidos pelo grupo.


In [ ]:
def criar_preprocessador_linear():
    return ColumnTransformer([
        ("nominais", OneHotEncoder(handle_unknown="ignore", sparse_output=False), variaveis_nominais),
        ("ordinais", StandardScaler(), variaveis_ordinais)
    ], remainder="drop")

def criar_preprocessador_arvores():
    return ColumnTransformer([
        ("nominais", OneHotEncoder(handle_unknown="ignore", sparse_output=False), variaveis_nominais),
        ("ordinais", "passthrough", variaveis_ordinais)
    ], remainder="drop")

modelos = {
    "Regressão Logística": Pipeline([
        ("preprocessamento", criar_preprocessador_linear()),
        ("classificador", LogisticRegression(max_iter=3000, random_state=SEMENTE))
    ]),
    "Árvore de Decisão": Pipeline([
        ("preprocessamento", criar_preprocessador_arvores()),
        ("classificador", DecisionTreeClassifier(random_state=SEMENTE))
    ]),
    "Random Forest": Pipeline([
        ("preprocessamento", criar_preprocessador_arvores()),
        ("classificador", RandomForestClassifier(random_state=SEMENTE, n_jobs=1))
    ])
}

print("Modelos configurados:", ", ".join(modelos))


## 7. Validação cruzada no treinamento

Será aplicado `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`. A cada rodada, o pipeline aprende a transformação e o classificador usando apenas quatro partes do treinamento e é avaliado na parte restante.

Registraremos **F1-macro médio e desvio-padrão**, **acurácia média e desvio-padrão** e F1-macro no treinamento. Uma diferença grande entre os escores de treino e validação pode indicar sobreajuste, mas deve ser interpretada juntamente com a pequena dimensão da base.

**Atenção:** a estratificação evita uma distribuição muito desigual entre as partes, mas não elimina a incerteza decorrente das classes raras.


In [ ]:
validacao = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEMENTE
)

pontuacoes = {}
for nome, pipeline in modelos.items():
    resultado = cross_validate(
        pipeline,
        X_treino, y_treino,
        cv=validacao,
        scoring={"f1_macro": "f1_macro", "acuracia": "accuracy"},
        return_train_score=True,
        n_jobs=1,
        error_score="raise"
    )
    pontuacoes[nome] = resultado
    print(f"Validação concluída: {nome}")


In [ ]:
linhas = []
for nome, resultado in pontuacoes.items():
    linhas.append({
        "modelo": nome,
        "f1_macro_medio": resultado["test_f1_macro"].mean(),
        "f1_macro_dp": resultado["test_f1_macro"].std(ddof=1),
        "acuracia_media": resultado["test_acuracia"].mean(),
        "acuracia_dp": resultado["test_acuracia"].std(ddof=1),
        "f1_treino_medio": resultado["train_f1_macro"].mean()
    })

resumo_cv = pd.DataFrame(linhas).sort_values(
    "f1_macro_medio", ascending=False
).reset_index(drop=True)

display(resumo_cv.round(3))


### 7.1 Comparação gráfica

A figura compara os três modelos com barras de erro correspondentes ao **desvio-padrão entre as cinco partes**. Essas barras descrevem a variação observada nas divisões; não equivalem a intervalos de confiança nem garantem o desempenho em novos contextos.


In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 4.7), sharey=True)

indicadores = [
    ("f1_macro_medio", "f1_macro_dp", "F1-macro"),
    ("acuracia_media", "acuracia_dp", "Acurácia")
]
for eixo, (media, desvio, titulo) in zip(eixos, indicadores):
    eixo.barh(
        resumo_cv["modelo"], resumo_cv[media],
        xerr=resumo_cv[desvio],
        color="#3869a6", alpha=0.90, capsize=4
    )
    eixo.set_xlim(0, 1)
    eixo.set_xlabel("Pontuação")
    eixo.set_title(f"{titulo}: média ± desvio-padrão")

eixos[0].invert_yaxis()

fig.suptitle("Comparação dos classificadores — validação cruzada (treinamento)")
fig.tight_layout()
plt.show()


## 8. Interpretação da validação e decisão do grupo

**Responder somente após conferir os resultados produzidos pelas células anteriores.**

1. Qual modelo apresentou maior F1-macro médio? Qual foi a diferença para os demais?
2. A acurácia reforça ou contradiz a ordenação do F1-macro?
3. Os desvios-padrão sugerem estabilidade ou alta variação entre as cinco partes?
4. A diferença entre F1 de treinamento e F1 de validação sugere possível sobreajuste?
5. A simplicidade e a interpretabilidade influenciam a decisão em caso de desempenhos próximos?

**Registro do grupo:**

- Modelo escolhido: **[preencher após validação]**
- Justificativa numérica: **[citar média, dispersão e comparação]**
- Participantes da validação: **[informar]**
- Data e evidência: **[informar]**

O código abaixo informa o primeiro lugar por F1-macro **apenas como referência**. Ele não aprova automaticamente a escolha.


In [ ]:
modelo_mais_bem_pontuado_cv = resumo_cv.loc[0, "modelo"]
print("Maior F1-macro médio na validação:", modelo_mais_bem_pontuado_cv)
print("Essa indicação não substitui a decisão fundamentada do grupo.")

# Depois da reunião e da análise dos resultados, escrever
# exatamente um dos nomes existentes no dicionário 'modelos'.
# Manter None até a decisão ser registrada.
MODELO_APROVADO = None

if MODELO_APROVADO is None:
    print("Avaliação final do teste bloqueada até a aprovação do grupo.")
else:
    assert MODELO_APROVADO in modelos, "Nome do modelo não reconhecido."


## 9. Avaliação única no teste reservado

**Executar esta seção apenas depois da decisão documentada na seção 8.** Alterar `MODELO_APROVADO` de `None` para o nome aprovado, executar novamente a seção 8 e, então, esta seção.

O modelo escolhido será ajustado sobre **todo o treinamento** e avaliado uma única vez no teste reservado. Serão registrados F1-macro, acurácia, precisão/recall/F1 de cada classe e matriz de confusão. Não usar esse resultado para retornar à seleção entre os três classificadores.

Com aproximadamente 29 observações de teste distribuídas por oito classes, pequenas mudanças na previsão podem alterar consideravelmente as métricas.


In [ ]:
modelo_final = None
previsoes_teste = None

if MODELO_APROVADO is None:
    print("Teste ainda não executado: aguarde a aprovação do grupo.")
else:
    modelo_final = clone(modelos[MODELO_APROVADO])
    modelo_final.fit(X_treino, y_treino)
    previsoes_teste = modelo_final.predict(X_teste)

    print("Modelo avaliado:", MODELO_APROVADO)
    print("F1-macro (teste):", round(f1_score(y_teste, previsoes_teste, average="macro", zero_division=0), 3))
    print("Acurácia (teste):", round(accuracy_score(y_teste, previsoes_teste), 3))
    print("\nRelatório por classe:")
    print(classification_report(
        y_teste, previsoes_teste,
        labels=list(rotulos_grade),
        target_names=list(rotulos_grade.values()),
        zero_division=0
    ))


In [ ]:
if previsoes_teste is None:
    print("Matriz de confusão indisponível até a avaliação autorizada.")
else:
    fig, ax = plt.subplots(figsize=(8.5, 7))
    ConfusionMatrixDisplay.from_predictions(
        y_teste, previsoes_teste,
        labels=list(rotulos_grade),
        display_labels=list(rotulos_grade.values()),
        cmap="Blues", colorbar=False, ax=ax
    )
    ax.set_title(f"Matriz de confusão no teste — {MODELO_APROVADO}")
    fig.tight_layout()
    plt.show()


## 10. Interpretação dos erros e incertezas

**Preencher depois do teste, com números e evidências verificadas:**

| Pergunta de análise | Registro do grupo |
|---|---|
| F1-macro e acurácia no teste | [informar valores calculados] |
| Classes com maior dificuldade | [identificar pelo relatório e pela matriz] |
| Erros mais recorrentes | [apontar células da matriz] |
| Diferenças em relação à validação | [comparar sem reajustar a escolha] |
| Incerteza decorrente de classes raras | [explicar considerando o suporte por classe] |
| Possíveis vieses e limitações | [fundamentar com os dados e a literatura] |

**Cuidado:** a proximidade numérica entre classes de `grade` não significa que todo erro tenha a mesma consequência acadêmica. A interpretação deve mencionar quais classes foram confundidas e quais consequências isso teria em um sistema de acompanhamento.


## 11. Limitações, ética, privacidade e aplicabilidade

1. **Dimensão da amostra:** 145 estudantes e oito classes, com poucos exemplos nas classes minoritárias. As métricas podem variar substancialmente entre divisões.
2. **Contexto original:** dados coletados em um cenário educacional específico. Não é possível generalizar automaticamente o resultado para alunos da UNIBALSAS ou do Brasil.
3. **Natureza dos atributos:** várias respostas são autorrelatadas ou categorias codificadas. A representação numérica de ordinais é uma hipótese metodológica.
4. **Vieses e grupos:** idade, sexo, renda e escolaridade familiar podem capturar desigualdades. O uso dessas características precisa ser justificado, com análise crítica e atenção à LGPD; grupos pequenos não permitem conclusões confiáveis sobre equidade.
5. **Explicabilidade:** regras de uma árvore podem ser mais fáceis de examinar que muitas árvores agregadas, mas interpretabilidade não substitui avaliação de desempenho.
6. **Relação com o MVP:** uma previsão de nota pode subsidiar acompanhamento acadêmico, **não** prioriza automaticamente tarefas. O MVP ainda requer dados de prazos, pesos, esforço, disciplina e situação das atividades.

**Posicionamento responsável:** um classificador acadêmico deve apoiar reflexão e acompanhamento, não rotular definitivamente estudantes nem decidir sozinho sobre oportunidades educacionais. Não incluir informações pessoais identificáveis na demonstração.


## 12. Integração com o Documento Técnico v3.0 e README

O relatório técnico deve **preservar** introdução, problema, fonte de dados, justificativas, limpeza e análise exploratória das TEDs 01 e 02. Acrescentar, com base nos resultados deste notebook:

1. Justificativa da classificação multiclasse de `grade`.
2. Separação estratificada, pré-processamento e prevenção de vazamento.
3. Descrição dos três pipelines e da validação cruzada.
4. Tabela e gráfico comparativo das métricas calculadas.
5. Decisão fundamentada do grupo **antes** do teste.
6. Avaliação final, matriz de confusão e análise dos erros.
7. Limitações, vieses, relação com o MVP e recomendações de evolução.

**README:** depois da validação do grupo, adicionar o link deste notebook, as configurações efetivamente utilizadas e um resumo fiel das métricas. Não inserir resultados provisórios como conclusões.

**Entregáveis ainda dependentes da equipe:** notebook executado e revisado; Documento Técnico v3.0; evidências de validação; eventual modelo salvo, se necessário ao MVP.


## 13. Registro de autoria, execução e validação

| Campo | Preenchimento pela equipe |
|---|---|
| Responsável pela execução do notebook | [informar] |
| Ambiente e versões utilizadas | [registrar saída da seção 3] |
| Data da execução | [informar] |
| Responsáveis pela análise dos resultados | [informar] |
| Justificativa da escolha do modelo | [registrar antes da avaliação final] |
| Conferência das métricas e gráficos | [descrever como foi realizada] |
| Validação da matriz de confusão | [informar] |
| Revisão do Documento Técnico v3.0 | [informar] |
| Commit/revisão final do GitHub | [registrar após entrega] |

### Declaração de apoio por inteligência artificial

**Ferramenta:** ChatGPT.  
**Finalidade:** orientação metodológica, organização do notebook, consulta de sintaxe e revisão técnica.  
**Partes apoiadas:** estrutura das células, construção dos pipelines, verificação das métricas e roteiro de documentação.  
**Decisões do grupo:** alvo `grade`, 31 preditoras, algoritmos, divisão 80/20, cinco partes, semente 42, métricas e representação mista.  
**Validação:** [registrar execução independente, conferência de resultados e decisões dos estudantes].  
**Privacidade:** não inserir dados pessoais, sensíveis ou sigilosos no histórico de consultas.

O histórico de uso da IA deverá ser disponibilizado conforme as orientações da disciplina.


## Referências e materiais de consulta

- YILMAZ, N.; ŞEKEROĞLU, B. *Higher Education Students Performance Evaluation*. UCI Machine Learning Repository. DOI: [10.24432/C51G82](https://doi.org/10.24432/C51G82).
- SCIKIT-LEARN DEVELOPERS. [User Guide — Pipelines and composite estimators](https://scikit-learn.org/stable/modules/compose.html).
- SCIKIT-LEARN DEVELOPERS. [Cross-validation: evaluating estimator performance](https://scikit-learn.org/stable/modules/cross_validation.html).
- SCIKIT-LEARN DEVELOPERS. [Classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics).
- Equipe StudyTrack IA. [TED 01](https://github.com/Francisco-alt-coder/StudyTrack_IA/blob/main/TED_01_Big_Data_StudyTrack_IA.pdf) e [TED 02](https://github.com/Francisco-alt-coder/StudyTrack_IA/blob/main/TED02_StudyTrack_.pdf), documentos do projeto acadêmico.

**Observação:** as conclusões, os valores de desempenho e a decisão final não foram preenchidos antecipadamente. Eles exigem execução, análise crítica e confirmação pelo grupo.
